# RF-DETR: Mammography Lesion Detection Quickstart

This notebook demonstrates the end-to-end workflow for RF-DETR on Google Colab:
1. Automated environment setup & compilation of Detrex CUDA operators
2. Automated test verification
3. Training with micro-batch gradient accumulation & mixed precision
4. TensorBoard visualization
5. Standalone COCO evaluation

### Hardware Requirement:
- GPU Runtime: NVIDIA T4, V100, A100, or L4 (Compute capability >= 7.0)

In [ ]:
# Step 1: Automated Environment Setup & Compilation
# Clones Detrex, installs Detectron2, compiles MSDeformAttn, and installs RF-DETR
!bash scripts/setup_colab.sh

In [ ]:
# Step 2: Verification Test Suite
# Runs automated offline tests on synthetic data
!pytest tests/ -v

In [ ]:
# Step 3: Dataset Setup (Optional: Mount Google Drive or set custom dataset path)
import os

DATA_DIR = "/content/mammo_data/images"
TRAIN_JSON = "/content/mammo_data/coco/mass_train.json"
VAL_JSON = "/content/mammo_data/coco/mass_val.json"
DINOV2_WEIGHTS = "/content/dinov2_latest_checkpoint.pth"
OUTPUT_DIR = "/content/drive/MyDrive/RF_DETR_Run1"

print(f"Data Dir: {DATA_DIR}")
print(f"Output Dir: {OUTPUT_DIR}")

In [ ]:
# Step 4: Launch Training via CLI
!python train.py \
    --config-file configs/mammo_dinov2_dino.py \
    --data-dir "$DATA_DIR" \
    --train-json "$TRAIN_JSON" \
    --val-json "$VAL_JSON" \
    --dinov2-weights "$DINOV2_WEIGHTS" \
    --output-dir "$OUTPUT_DIR" \
    --batch-size 2 \
    --accum-steps 8 \
    --epochs 12 \
    --lr 1e-4 \
    --amp auto

In [ ]:
# Step 5: Launch TensorBoard Monitoring
%load_ext tensorboard
%tensorboard --logdir "$OUTPUT_DIR/tensorboard"

In [ ]:
# Step 6: Standalone COCO Evaluation
CHECKPOINT_PATH = f"{OUTPUT_DIR}/best_checkpoint.pth"

!python eval.py \
    --config-file configs/mammo_dinov2_dino.py \
    --checkpoint "$CHECKPOINT_PATH" \
    --data-dir "$DATA_DIR" \
    --eval-json "$VAL_JSON" \
    --output-dir "$OUTPUT_DIR/eval_results"